# Paper 2 — Embedding Divergence: Complete Control

**Runtime: CPU, ~30–45 minutes total. Downloads `all-MiniLM-L6-v2` (needs
internet; cached after the first part).**

This is the **whole** divergence control, not just the headline test. Run all
four parts — the base test on its own reports a number that one of the later
parts shows to be flawed.

## Why four parts

Trajectory divergence is the cosine distance between the centroid of the
sentences *before* a marker and the centroid of those *after* it. The threat
is that reasoning text drifts semantically as it proceeds regardless of
revisions — so if divergence at an arbitrary point matches divergence at a
revision, it is measuring generic drift, not revision structure.

| Part | What it does | Why it is here |
|---|---|---|
| **1. Three-arm control** | REVISION vs matched FILLER vs RANDOM positions | the base test |
| **2. Confound audit (C1, C2)** | fixes an arm-asymmetric length filter | **the base test's decisive number is not trustworthy without this** |
| **3. Asymmetry checks (C3, C4, C5)** | position, and two further structural asymmetries, individually and together | the revision-vs-random null is load-bearing, so these needed closing rather than assuming |
| **4. Gap diagnostic** | why the FILLER arm sits low | the actual explanation of the one apparent effect |

## The flaw part 2 exists to fix

In part 1 the random arm draws only from traces with
`len(full_response) >= 2000`, because it needs room to sample a midpoint. The
revision and filler arms have no length constraint. So the decisive result —
revision vs random — is a comparison between **differently selected trace
populations**, not between positions within the same population. Part 2
re-runs it with the filter applied symmetrically.

Report part 2's corrected numbers, not part 1's.

## What the parts together establish

Revision divergence is **indistinguishable from arbitrary positions** in the
same trace, under every variant tested. Fillers sit well *below* both, so the
one apparent effect is not "revisions are high" but "fillers are low" — and
part 4 traces that to local repetition rather than to revision structure.

## Not included, deliberately

The kill-screen AUC (divergence scored against the human local/global labels)
and the marker-restricted scope recovery are retired and live in
`scope_analysis/`. The three-arm test here uses **no scope labels at all** —
its arms are defined by revision status and position — which is why it
survives the scope cut intact.

**Prerequisites on Drive:** `labels/`, `rejects/`, `traces/`.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# Part 1 contains one optional cross-check against the retired scope-labeled
# event set. The three-arm test itself uses NO scope labels.
SCOPE_ANALYSIS_ENABLED = False   # leave False unless reproducing retired work

import sys, types
_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False
def _b():
    if _sf.SCOPE_BANNER_SHOWN: return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    print(f"\n[scope_flag] SCOPE_ANALYSIS_ENABLED={_sf.SCOPE_ANALYSIS_ENABLED}")
    return _sf.SCOPE_ANALYSIS_ENABLED
def _s(n):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {n}")
_sf.scope_banner = _b; _sf.skip_note = _s
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

> **Run the four parts in order, top to bottom, and do not re-run an earlier
> part afterwards.** Each part is a separate script and all four define
> functions under the same names (`main`, `describe`, `divergence`), so each
> part redefines the previous one's. That is correct in order — the earlier
> part has already finished — but re-running part 1's final cell after part 3
> would execute part 3's `main()`. Re-run from the top if you need to redo an
> earlier part.
>
> Each part loads the embedding model itself. The first download takes a
> minute; afterwards it is cached for the session.


---

# Part 1 — Three-arm control

REVISION vs matched FILLER vs RANDOM. Establishes the base comparison.

**Treat the revision-vs-random number printed here as provisional** — part 2 re-runs it without the arm-asymmetric length filter. The optional cross-check against the scope-labeled event set is gated off by default.


In [ ]:
# =============================================================================
# EMBEDDING DIVERGENCE CONTROL
#
# THE THREAT
# Trajectory divergence = cosine distance between the centroid of the sentences
# before a revision and the centroid of the sentences after it. But reasoning
# text drifts semantically as it proceeds REGARDLESS of revisions. If divergence
# at an arbitrary point in a trace is the same as at a revision, then divergence
# is measuring generic semantic drift, not anything about revisions -- exactly
# the failure mode the filler control exposed for causal necessity.
#
# THREE ARMS, identical extraction and metric:
#   REVISION - confirmed revision positions (labels/*.json, verified=true)
#   FILLER   - GPT-4o-verified NON-revision positions with the SAME marker words
#              (rejects/*_confirmed_filler.json), matched 1:1 on (model, marker).
#              This is the direct analogue of the necessity filler control.
#   RANDOM   - arbitrary positions in the same traces, no marker at all.
#              Establishes the baseline drift rate for text with no discourse
#              marker present, which the filler arm cannot isolate (fillers
#              still contain a marker word).
#
# INTERPRETATION
#   revision > filler ~= random  -> divergence responds to genuine revisions.
#   revision ~= filler > random  -> divergence responds to MARKER WORDS, not
#                                   revision status. Would mean the scope
#                                   result rests on a lexical artifact.
#   all three equal              -> divergence is generic drift. The scope
#                                   result would have to be withdrawn, as the
#                                   necessity finding was.
#
# WHY REBUILD THE REVISION ARM
# The original divergences came from annotation-workbook context; controls must
# come from raw traces. Comparing those directly would confound extraction
# differences with the effect. So all three arms are rebuilt here from raw
# traces with identical logic, replicating the workbook's own windowing
# (900/700 char budgets, then 6-sentence caps). The script cross-checks the
# rebuilt revision arm against the original kill-screen values.
#
# COST: CPU + internet for the embedding model. No GPU, minutes not hours.
# =============================================================================



from pathlib import Path
from collections import defaultdict, Counter
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

import scope_flag   # SCOPE_ANALYSIS_ENABLED gate; see SCOPE_ISOLATION.md

In [ ]:
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
REJECT_DIR  = GDRIVE_BASE / "rejects"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"

KILL_EVENTS_PATH = EXPORT_DIR / "kill_screen_events.json"   # for the cross-check
OUT_PATH         = EXPORT_DIR / "divergence_control.json"

In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED = 8080

# replicate the annotation-workbook windowing exactly
CHAR_BUDGET_BEFORE = 900
CHAR_BUDGET_AFTER  = 700
RAW_BUFFER_BEFORE  = 2500
RAW_BUFFER_AFTER   = 1200
N_BEFORE = N_AFTER = 6      # then the kill-screen sentence caps

In [ ]:
MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]


# ---------------------------------------------------------------- helpers ---
def clean_bpe(text):
    return text.replace("\u0120", " ").replace("\u010a", "\n")


def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p.strip()]


def take_by_budget(sentences, budget, from_end):
    if not sentences:
        return []
    ordered = list(reversed(sentences)) if from_end else sentences
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget:
            break
    return list(reversed(out)) if from_end else out


def normalize_marker(text):
    return re.sub(r"[^\w\s]", "", str(text).lower()).strip()


def extract_window(full_response, char_start, char_end):
    """Replicates build_blind_sample_xlsx.py's windowing, then the kill-screen
    6-sentence caps. Identical for all three arms."""
    raw_start = max(0, char_start - RAW_BUFFER_BEFORE)
    raw_end   = min(len(full_response), char_end + RAW_BUFFER_AFTER)
    window = clean_bpe(full_response[raw_start:raw_end])

    local_start = char_start - raw_start
    local_end   = char_end - raw_start
    if not (0 <= local_start <= local_end <= len(window)):
        return None, None

    before_txt = window[:local_start]
    after_txt  = window[local_end:]

    before = take_by_budget(split_sentences(before_txt), CHAR_BUDGET_BEFORE, from_end=True)
    after  = take_by_budget(split_sentences(after_txt),  CHAR_BUDGET_AFTER,  from_end=False)
    # drop a trailing fragment cut off by the raw buffer
    if after and not re.search(r'[.!?]\s*$', after[-1]):
        after = after[:-1]

    before = before[-N_BEFORE:]
    after  = after[:N_AFTER]
    if not before or not after:
        return None, None
    return before, after


def load_trace(model, pid, cache):
    key = (model, pid)
    if key not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[key] = json.load(open(p)) if p.exists() else None
    return cache[key]

In [ ]:
# ---------------------------------------------------------- build the arms ---
def build_revision_arm(cache):
    out = []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if not mdir.exists():
            continue
        for f in mdir.glob("*.json"):
            obj = json.load(open(f))
            pid = obj.get("problem_id", f.stem)
            revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
            if not revs:
                continue
            trace = load_trace(model, pid, cache)
            if trace is None or not trace.get("correct", False) or trace.get("truncated", False):
                continue
            fr = trace.get("full_response", "")
            for r in revs:
                cs, ce = r.get("char_start"), r.get("char_end")
                if cs is None or ce is None or not (0 <= cs < len(fr)):
                    continue
                b, a = extract_window(fr, cs, ce)
                if b is None:
                    continue
                out.append({"arm": "revision", "model": model, "problem_id": pid,
                            "marker": normalize_marker(r.get("matched_text", "")),
                            "before": b, "after": a})
    return out


def build_filler_arm(cache):
    out = []
    for model in MODELS:
        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if not rp.exists():
            continue
        for flr in json.load(open(rp)):
            pid = flr["problem_id"]
            trace = load_trace(model, pid, cache)
            if trace is None or not trace.get("correct", False) or trace.get("truncated", False):
                continue
            fr = trace.get("full_response", "")
            cs = flr.get("char_start")
            matched = flr.get("matched_text", "")
            if cs is None or not (0 <= cs < len(fr)):
                continue
            ce = cs + len(matched)
            b, a = extract_window(fr, cs, min(ce, len(fr)))
            if b is None:
                continue
            out.append({"arm": "filler", "model": model, "problem_id": pid,
                        "marker": normalize_marker(matched),
                        "before": b, "after": a})
    return out


def build_random_arm(cache, per_model_counts, rng):
    """Arbitrary positions in the same traces -- no marker present. Zero-width
    'event' so before/after split at a plain character offset."""
    out = []
    for model, n_wanted in per_model_counts.items():
        pids = [pid for (m, pid), tr in cache.items()
                if m == model and tr is not None
                and tr.get("correct", False) and not tr.get("truncated", False)]
        if not pids:
            continue
        attempts, made = 0, 0
        while made < n_wanted and attempts < n_wanted * 20:
            attempts += 1
            pid = pids[rng.randrange(len(pids))]
            fr = cache[(model, pid)].get("full_response", "")
            if len(fr) < 2000:
                continue
            cs = rng.randrange(1000, len(fr) - 800)
            b, a = extract_window(fr, cs, cs)   # zero-width split point
            if b is None:
                continue
            out.append({"arm": "random", "model": model, "problem_id": pid,
                        "marker": "", "before": b, "after": a})
            made += 1
    return out


# ------------------------------------------------------------------- stats ---
def describe(vals, label):
    v = np.array(vals)
    print(f"\n  {label}  (n={len(v)})")
    print(f"    mean {v.mean():.4f}   median {np.median(v):.4f}   sd {v.std():.4f}")
    print(f"    range [{v.min():.4f}, {v.max():.4f}]")
    return v


def compare(a, b, la, lb):
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    auc = u / (len(a) * len(b))
    rng = np.random.RandomState(0)
    diffs = [np.median(rng.choice(a, len(a), True)) - np.median(rng.choice(b, len(b), True))
             for _ in range(5000)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"\n  {la} vs {lb}:")
    print(f"    Mann-Whitney p = {p:.4f}   P({la} > {lb}) = {auc:.3f}")
    print(f"    median difference = {np.median(a)-np.median(b):+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
    return {"p": float(p), "auc": float(auc),
            "median_diff": float(np.median(a) - np.median(b)), "ci95": [float(lo), float(hi)]}


def main():
    cache = {}
    print("Building revision arm...")
    rev = build_revision_arm(cache)
    print(f"  {len(rev)} revision events")

    print("Building filler arm...")
    flr_pool = build_filler_arm(cache)
    print(f"  {len(flr_pool)} eligible filler positions")

    # match filler 1:1 to revision on (model, marker)
    rng = random.Random(SEED)
    by_cell = defaultdict(list)
    for e in flr_pool:
        by_cell[(e["model"], e["marker"])].append(e)
    demand = Counter((e["model"], e["marker"]) for e in rev)
    flr, unmatched = [], 0
    for cell, n_needed in demand.items():
        avail = by_cell.get(cell, [])
        take = min(n_needed, len(avail))
        if take < n_needed:
            unmatched += n_needed - take
        if take:
            flr.extend(rng.sample(avail, take))
    print(f"  matched {len(flr)} filler controls ({unmatched} revision events unmatched)")

    print("Building random arm...")
    rnd = build_random_arm(cache, Counter(e["model"] for e in rev), rng)
    print(f"  {len(rnd)} random positions")

    # ---------------------------------------------------------- embed ------
    print(f"\nLoading {EMBED_MODEL_NAME} ...")
    embedder = SentenceTransformer(EMBED_MODEL_NAME)

    def divergence(ev):
        b = embedder.encode(ev["before"], normalize_embeddings=True)
        a = embedder.encode(ev["after"],  normalize_embeddings=True)
        cb, ca = b.mean(axis=0), a.mean(axis=0)
        return 1.0 - float(np.dot(cb, ca) / (np.linalg.norm(cb) * np.linalg.norm(ca)))

    print("Computing divergence for all arms...")
    for arm in (rev, flr, rnd):
        for ev in arm:
            ev["divergence"] = divergence(ev)

    # ------------------------------------------- cross-check vs original ---
    # The THREE-ARM TEST ITSELF uses no scope labels -- revision/filler/random
    # arms are defined by revision status and position, never by local/global.
    # The three-arm result is therefore KEPT and survives the scope cut intact.
    # Only this optional cross-check reads the scope-labeled event set, so it
    # is gated. Its input now lives in scope_analysis/kill_screen_events.json.
    if scope_flag.scope_banner() and KILL_EVENTS_PATH.exists():
        orig = json.load(open(KILL_EVENTS_PATH))
        orig_div = [divergence({"before": e["before_sents"], "after": e["after_sents"]})
                    for e in orig]
        rev_div_all = [e["divergence"] for e in rev]
        print(f"\n{'='*70}\nCROSS-CHECK: rebuilt revision arm vs. original kill-screen events\n{'='*70}")
        print(f"  original kill-screen events (n={len(orig_div)}): "
              f"median {np.median(orig_div):.4f}, mean {np.mean(orig_div):.4f}")
        print(f"  rebuilt revision arm (n={len(rev_div_all)}): "
              f"median {np.median(rev_div_all):.4f}, mean {np.mean(rev_div_all):.4f}")
        print("  (These use different event sets -- the rebuilt arm covers ALL confirmed")
        print("   revisions, the original only the human-labeled subset -- so expect")
        print("   similar but not identical distributions. A large gap would indicate the")
        print("   windowing replication is wrong and the comparison below is unsafe.)")

    # ------------------------------------------------------------ results --
    print(f"\n{'='*70}\nDIVERGENCE BY ARM\n{'='*70}")
    v_rev = describe([e["divergence"] for e in rev], "REVISION positions")
    v_flr = describe([e["divergence"] for e in flr], "FILLER positions (same markers, verified non-revision)")
    v_rnd = describe([e["divergence"] for e in rnd], "RANDOM positions (no marker)")

    print(f"\n{'='*70}\nARM COMPARISONS\n{'='*70}")
    res = {}
    res["revision_vs_filler"] = compare(v_rev, v_flr, "revision", "filler")
    res["revision_vs_random"] = compare(v_rev, v_rnd, "revision", "random")
    res["filler_vs_random"]   = compare(v_flr, v_rnd, "filler",   "random")

    print(f"\n  per-model medians:")
    for m in MODELS:
        r_ = [e["divergence"] for e in rev if e["model"] == m]
        f_ = [e["divergence"] for e in flr if e["model"] == m]
        d_ = [e["divergence"] for e in rnd if e["model"] == m]
        fmt = lambda x: f"{np.median(x):.4f}" if x else "  -   "
        print(f"    {m:<32} rev {fmt(r_)} (n={len(r_):>4})  "
              f"fil {fmt(f_)} (n={len(f_):>4})  rnd {fmt(d_)} (n={len(d_):>4})")

    # --------------------------------------------------------------- read --
    print(f"\n{'#'*70}\n# READ\n{'#'*70}")
    p_rf = res["revision_vs_filler"]["p"]
    p_rr = res["revision_vs_random"]["p"]
    p_fr = res["filler_vs_random"]["p"]
    ci_rr = res["revision_vs_random"]["ci95"]
    med_rev, med_flr, med_rnd = np.median(v_rev), np.median(v_flr), np.median(v_rnd)

    beats_filler = p_rf < 0.05 and med_rev > med_flr
    beats_random = p_rr < 0.05 and med_rev > med_rnd

    # RANDOM is the decisive control. Filler positions still contain a marker
    # word AND are, by GPT-4o's own definition, points where no genuine change
    # occurs -- so they may sit at unusually CONTINUOUS points in the reasoning,
    # making them an artificially low baseline rather than a neutral one.
    # Beating filler alone therefore does NOT establish revision-sensitivity;
    # beating arbitrary text does. Both must hold.
    if beats_filler and beats_random:
        print("Divergence exceeds BOTH controls -- matched fillers and random positions.")
        print("It responds to revision status rather than to marker words or generic")
        print("drift -- i.e. divergence is a revision-sensitive measurement.")

    elif beats_filler and not beats_random:
        print("SPLIT RESULT -- divergence beats fillers but NOT random positions.")
        print(f"  revision vs filler : +{med_rev-med_flr:.4f}  (p={p_rf:.4g})")
        print(f"  revision vs random : {med_rev-med_rnd:+.4f}  (p={p_rr:.4g}, "
              f"95% CI [{ci_rr[0]:+.4f}, {ci_rr[1]:+.4f}])")
        print()
        print("Divergence at a revision is NOT distinguishable from divergence at an")
        print("arbitrary point in the same trace. It therefore does not identify")
        print("revisions against a generic-text baseline, and is not usable as evidence")
        print("that a semantic shift occurs at revisions.")
        print()
        print("The revision-filler gap is most likely explained by fillers being")
        print("unusually CONTINUOUS -- they are, by construction, marker words at points")
        print("where no genuine change occurs -- rather than by revisions being")
        print("unusually divergent. Note filler also sits BELOW random "
              f"({med_flr:.4f} vs {med_rnd:.4f}), which is consistent with that reading.")
        print()
        print("Divergence is not inert (it separates revisions from discourse padding,")
        print("robustly and in every model), but what it tracks is not 'a semantic shift")
        print("is happening here'. Report the random-arm null alongside any use of it.")

    elif not beats_filler and p_fr < 0.05 and med_flr > med_rnd:
        print("Divergence does NOT distinguish revisions from same-marker fillers, but")
        print("BOTH exceed random positions. This implicates the MARKER WORD, not")
        print("revision status -- divergence tracks the lexical/discourse environment")
        print("around these words rather than whether a genuine revision occurred.")
        print("Divergence would then be a lexical artifact -- a direct parallel to the")
        print("confound Task 1 was built to rule out.")

    elif p_rf >= 0.05 and p_rr >= 0.05:
        print("Divergence is indistinguishable across ALL THREE arms. It measures generic")
        print("semantic drift in reasoning text, not anything about revisions. Any claim")
        print("resting on embedding divergence must be withdrawn on the same grounds as")
        print("the causal-necessity finding.")

    else:
        print("Pattern not covered by the standard readings -- interpret the three")
        print("pairwise comparisons directly. Key question: does revision exceed RANDOM?")
        print(f"  revision vs random: {med_rev-med_rnd:+.4f}, p={p_rr:.4g}, "
              f"CI [{ci_rr[0]:+.4f}, {ci_rr[1]:+.4f}]")

    print("\nNote: RANDOM is the decisive control here, not FILLER. Filler positions")
    print("still contain a marker word and may be atypically continuous, so they are")
    print("not a neutral baseline. Check CI widths before calling any null conclusive.")

    json.dump({
        "n_revision": len(v_rev), "n_filler": len(v_flr), "n_random": len(v_rnd),
        "median_revision": float(np.median(v_rev)),
        "median_filler": float(np.median(v_flr)),
        "median_random": float(np.median(v_rnd)),
        "mean_revision": float(v_rev.mean()), "mean_filler": float(v_flr.mean()),
        "mean_random": float(v_rnd.mean()),
        "comparisons": res,
    }, open(OUT_PATH, "w"), indent=2)
    print(f"\nWrote {OUT_PATH}")


if __name__ == "__main__":
    main()

---

# Part 2 — Confound audit (C1, C2)

**This part produces the number to report.** C1 is the arm-asymmetric length filter described in the header: the random arm required `len(full_response) >= 2000` while the other two arms had no length constraint, so part 1's decisive comparison ran across differently-selected trace populations. Both problems were found only after the necessity audit, and were missed when the divergence work was first written up.


In [ ]:
# =============================================================================
# DIVERGENCE CONFOUND AUDIT
#
# Two problems were found in divergence_control.py after the necessity audit.
# Both were missed when the divergence work was first written up.
#
#   C1  ARM-ASYMMETRIC LENGTH FILTER. The random arm draws only from traces
#       with len(full_response) >= 2000 (needed to sample a midpoint), while
#       the revision and filler arms have no length constraint. The decisive
#       result -- revision vs random, p = 0.337, CI [-0.005, +0.014] -- is
#       therefore a comparison between differently-selected trace populations.
#       Longer traces plausibly permit more topic drift, which would inflate
#       the random arm and mask a real revision effect.
#
#   C2  CORRUPTED ELIGIBILITY FLAG. All three arms filter on trace["correct"],
#       which the ground-truth audit found wrong for 195 model-problem pairs
#       (184 false negatives, 11 false positives). The divergence pools were
#       never checked against the false-positive list -- only the necessity
#       pools were.
#
#   C3  POSITION UNCHECKED FOR RANDOM. The gap diagnostic found revision and
#       filler differ in relative trace position (0.477 vs 0.515, p = 0.0006).
#       The random arm's positional distribution was never compared, even
#       though random positions are drawn uniformly in character space and so
#       need not resemble where revisions actually occur.
#
# THE TEST THAT MATTERS is C1: re-run revision vs random with the >=2000
# constraint applied SYMMETRICALLY. If the null survives, the published
# conclusion stands. If it does not, the headline divergence result needs
# revisiting.
#
# COST: CPU + internet for the embedding model. Minutes.
# =============================================================================



from pathlib import Path
from collections import defaultdict
import json
import random
import re

import numpy as np
from scipy.stats import mannwhitneyu

In [ ]:
from sentence_transformers import SentenceTransformer

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR = GDRIVE_BASE / "labels"
TRACES_DIR = GDRIVE_BASE / "traces"
REJECT_DIR = GDRIVE_BASE / "rejects"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
REPAIRED_GT = EXPORT_DIR / "repaired_ground_truth.json"

In [ ]:
OUT = EXPORT_DIR / "divergence_confound_audit.json"

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED = 31337

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE, RAW_BUFFER_AFTER = 2500, 1200
N_BEFORE = N_AFTER = 6
MIN_LEN_FOR_RANDOM = 2000      # the asymmetric constraint under investigation

In [ ]:
MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B", "DeepSeek-R1-Distill-Qwen-14B"]


def clean_bpe(t):
    return t.replace("\u0120", " ").replace("\u010a", "\n")


def split_sentences(text):
    return [p for p in re.split(r'(?<=[.!?])\s+', text.strip()) if p.strip()]


def take_by_budget(sents, budget, from_end):
    if not sents:
        return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s)
        total += len(s) + 1
        if total >= budget:
            break
    return list(reversed(out)) if from_end else out


def normalise_marker(t):
    return re.sub(r"[^\w\s]", "", str(t).lower()).strip()


def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE)
    re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_])
    ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)):
        return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]):
        after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after:
        return None, None
    return before, after


def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

In [ ]:
def eligible(tr):
    return tr is not None and tr.get("correct", False) and not tr.get("truncated", False)


def build_arms(cache, rng):
    """Same construction as divergence_control.py, but records trace length and
    relative position on EVERY arm so the confounds can be measured."""
    rev, flr, rnd = [], [], []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if mdir.exists():
            for f in mdir.glob("*.json"):
                obj = json.load(open(f))
                pid = obj.get("problem_id", f.stem)
                revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
                if not revs:
                    continue
                tr = load_trace(model, pid, cache)
                if not eligible(tr):
                    continue
                fr = tr.get("full_response", "")
                for r in revs:
                    cs, ce = r.get("char_start"), r.get("char_end")
                    if cs is None or ce is None or not (0 <= cs < len(fr)):
                        continue
                    b, a = extract_window(fr, cs, ce)
                    if b is None:
                        continue
                    rev.append({"arm": "revision", "model": model, "problem_id": pid,
                                "marker": normalise_marker(r.get("matched_text", "")),
                                "before": b, "after": a, "trace_len": len(fr),
                                "rel_pos": cs / max(1, len(fr))})
        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if rp.exists():
            for e in json.load(open(rp)):
                pid = e["problem_id"]
                tr = load_trace(model, pid, cache)
                if not eligible(tr):
                    continue
                fr = tr.get("full_response", "")
                cs, matched = e.get("char_start"), e.get("matched_text", "")
                if cs is None or not (0 <= cs < len(fr)):
                    continue
                b, a = extract_window(fr, cs, min(cs + len(matched), len(fr)))
                if b is None:
                    continue
                flr.append({"arm": "filler", "model": model, "problem_id": pid,
                            "marker": normalise_marker(matched),
                            "before": b, "after": a, "trace_len": len(fr),
                            "rel_pos": cs / max(1, len(fr))})

    per_model = defaultdict(int)
    for e in rev:
        per_model[e["model"]] += 1
    for model, n_want in per_model.items():
        pids = [pid for (m, pid), tr in cache.items() if m == model and eligible(tr)]
        made = attempts = 0
        while made < n_want and attempts < n_want * 20:
            attempts += 1
            pid = pids[rng.randrange(len(pids))]
            fr = cache[(model, pid)].get("full_response", "")
            if len(fr) < MIN_LEN_FOR_RANDOM:      # <-- the asymmetric filter
                continue
            cs = rng.randrange(1000, len(fr) - 800)
            b, a = extract_window(fr, cs, cs)
            if b is None:
                continue
            rnd.append({"arm": "random", "model": model, "problem_id": pid,
                        "marker": "", "before": b, "after": a,
                        "trace_len": len(fr), "rel_pos": cs / max(1, len(fr))})
            made += 1
    return rev, flr, rnd


def mw(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 5 or len(b) < 5:
        return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    rng = np.random.RandomState(0)
    d = [np.median(rng.choice(a, len(a), True)) - np.median(rng.choice(b, len(b), True))
         for _ in range(3000)]
    return {"p": float(p), "auc": float(u / (len(a) * len(b))),
            "median_a": float(np.median(a)), "median_b": float(np.median(b)),
            "median_diff": float(np.median(a) - np.median(b)),
            "ci95": [float(x) for x in np.percentile(d, [2.5, 97.5])],
            "n_a": len(a), "n_b": len(b)}


def show(tag, r):
    if r is None:
        print(f"    {tag:<44} (insufficient n)")
        return
    star = "*" if r["p"] < 0.05 else " "
    print(f"    {tag:<44} {r['median_a']:.4f} vs {r['median_b']:.4f}  "
          f"diff {r['median_diff']:+.4f} CI[{r['ci95'][0]:+.4f},{r['ci95'][1]:+.4f}] "
          f"p={r['p']:.4f}{star} n={r['n_a']}/{r['n_b']}")


def load_false_positives():
    """Problems whose stored `correct` flag was wrong (marked correct, actually
    incorrect). These should never have been eligible -- the flag is the
    eligibility filter -- so they are dropped from every arm before analysis."""
    if not REPAIRED_GT.exists():
        return set()
    rep = json.load(open(REPAIRED_GT))
    return {tuple(k.split("|", 1)) for k, v in rep.items()
            if v.get("correct_old") and not v.get("correct_new")}


def main():
    results = {}
    rng = random.Random(SEED)
    cache = {}

    fps = load_false_positives()
    print(f"\nFalse positives from the ground-truth audit: {len(fps)}")
    print("These are excluded from all arms -- their `correct` flag, which is the")
    print("eligibility filter, was wrong, so they were never eligible.")
    print("\nBuilding arms...")
    rev, flr, rnd = build_arms(cache, rng)
    print(f"  before exclusion: revision {len(rev)}   filler {len(flr)}   random {len(rnd)}")

    def drop_fps(arm):
        keep = [e for e in arm if (e["model"], e["problem_id"]) not in fps]
        return keep, len(arm) - len(keep)

    rev, n_dr = drop_fps(rev)
    flr, n_df = drop_fps(flr)
    rnd, n_dn = drop_fps(rnd)
    print(f"  dropped as false positives: {n_dr} revision, {n_df} filler, {n_dn} random")
    print(f"  after  exclusion: revision {len(rev)}   filler {len(flr)}   random {len(rnd)}")
    results["excluded_false_positives"] = {
        "n_problems": len(fps), "dropped_revision": n_dr,
        "dropped_filler": n_df, "dropped_random": n_dn}

    # ---------------- C2: corrupted eligibility flag ------------------------
    print("\n" + "#" * 84)
    print("# C2 — DID FALSE-POSITIVE PROBLEMS ENTER THE DIVERGENCE ARMS?")
    print("#" * 84)
    fps = set()
    if REPAIRED_GT.exists():
        rep = json.load(open(REPAIRED_GT))
        fps = {k for k, v in rep.items() if v.get("correct_old") and not v.get("correct_new")}
    print(f"\n  false positives from the ground-truth audit: {len(fps)}")
    c2 = {}
    for name, arm in (("revision", rev), ("filler", flr), ("random", rnd)):
        keys = {f"{e['model']}|{e['problem_id']}" for e in arm}
        hit = sorted(keys & fps)
        c2[name] = {"n_unique_problems": len(keys), "n_false_positive": len(hit),
                    "which": hit}
        flag = "  <-- CONTAMINATED" if hit else ""
        print(f"  {name:<10} {len(keys):>4} unique problems, {len(hit)} false positives{flag}")
        for h in hit:
            print(f"      {h}")
    results["c2_false_positives"] = c2

    # ---------------- C1/C3: arm composition --------------------------------
    print("\n" + "#" * 84)
    print("# C1/C3 — ARM COMPOSITION: trace length and position")
    print("#" * 84)
    print(f"\n  {'arm':<10}{'n':>6}{'median len':>13}{'% >=2000':>11}{'median rel_pos':>16}")
    comp = {}
    for name, arm in (("revision", rev), ("filler", flr), ("random", rnd)):
        L = np.array([e["trace_len"] for e in arm], float)
        P = np.array([e["rel_pos"] for e in arm], float)
        frac = float((L >= MIN_LEN_FOR_RANDOM).mean())
        print(f"  {name:<10}{len(arm):>6}{np.median(L):>13.0f}{frac:>10.0%}{np.median(P):>16.3f}")
        comp[name] = {"n": len(arm), "median_len": float(np.median(L)),
                      "frac_ge_min": frac, "median_rel_pos": float(np.median(P))}
    results["c1_c3_composition"] = comp
    excl = 1 - comp["revision"]["frac_ge_min"]
    print(f"\n  {excl:.0%} of revision events sit in traces SHORTER than {MIN_LEN_FOR_RANDOM}")
    print(f"  chars. Those traces can never contribute a random-arm event, so the")
    print(f"  random arm is drawn from a systematically longer population.")

    # ---------------- embed and compare -------------------------------------
    print(f"\nLoading {EMBED_MODEL_NAME} ...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)

    def divergence(ev):
        b = emb.encode(ev["before"], normalize_embeddings=True)
        a = emb.encode(ev["after"], normalize_embeddings=True)
        cb, ca = b.mean(axis=0), a.mean(axis=0)
        return 1.0 - float(np.dot(cb, ca) / (np.linalg.norm(cb) * np.linalg.norm(ca)))

    print("Computing divergence...")
    for arm in (rev, flr, rnd):
        for e in arm:
            e["divergence"] = divergence(e)

    d_rev = [e["divergence"] for e in rev]
    d_flr = [e["divergence"] for e in flr]
    d_rnd = [e["divergence"] for e in rnd]

    print("\n" + "#" * 84)
    print("# THE DECISIVE TEST — revision vs random, before and after symmetric filtering")
    print("#" * 84 + "\n")
    as_published = mw(d_rev, d_rnd)
    show("as published (asymmetric length filter)", as_published)

    rev_long = [e["divergence"] for e in rev if e["trace_len"] >= MIN_LEN_FOR_RANDOM]
    n_excluded = len(rev) - len(rev_long)
    symmetric = mw(rev_long, d_rnd)
    show(f"SYMMETRIC (both arms, traces >={MIN_LEN_FOR_RANDOM})", symmetric)

    # A previous version of this audit reported these two rows without checking
    # whether the filter removed anything. It removed nothing, so the "symmetric"
    # row was the same comparison printed twice -- a tautology presented as a
    # robustness check. Say plainly whether the filter bites.
    print(f"\n    filter effect: {n_excluded}/{len(rev)} revision events excluded "
          f"({n_excluded / max(1, len(rev)):.1%})")
    if n_excluded == 0:
        print("    -> THE FILTER IS VACUOUS. Every revision event already sits in a trace")
        print(f"       of >={MIN_LEN_FOR_RANDOM} chars, so the symmetric row is identical to")
        print("       the row above by construction. C1 was a real code asymmetry but has")
        print("       NO practical effect: the constraint was already satisfied everywhere.")
        print("       The conclusion is supported by the constraint being vacuous, NOT by")
        print("       a robustness check -- do not describe it as the latter.")
    else:
        print(f"    -> the filter removes {n_excluded} events, so the symmetric row is a")
        print("       genuine robustness check. Compare the two rows.")
    results["decisive"] = {"as_published": as_published, "symmetric": symmetric,
                           "n_revision_excluded_by_filter": n_excluded,
                           "filter_is_vacuous": n_excluded == 0}

    # divergence vs trace length, to see whether the confound has teeth
    allev = rev + flr + rnd
    L = np.array([e["trace_len"] for e in allev], float)
    D = np.array([e["divergence"] for e in allev], float)
    from scipy.stats import spearmanr
    rho, prho = spearmanr(L, D)
    print(f"\n    trace length vs divergence: Spearman rho = {rho:+.3f}, p = {prho:.4g}")
    results["length_vs_divergence"] = {"rho": float(rho), "p": float(prho)}

    print("\n  supporting comparisons (symmetric where applicable):")
    flr_long = [e["divergence"] for e in flr if e["trace_len"] >= MIN_LEN_FOR_RANDOM]
    show("revision vs filler (as published)", mw(d_rev, d_flr))
    show("revision vs filler (both >=2000)", mw(rev_long, flr_long))
    show("filler vs random (symmetric)", mw(flr_long, d_rnd))

    # ---------------- read ---------------------------------------------------
    print("\n" + "#" * 84)
    print("# READ")
    print("#" * 84)
    if as_published and symmetric:
        vacuous = results["decisive"]["filter_is_vacuous"]
        was_null = as_published["p"] >= 0.05
        now_null = symmetric["p"] >= 0.05
        print(f"\n  revision vs random: p {as_published['p']:.4f} (as published) -> "
              f"{symmetric['p']:.4f} (symmetric)")
        if vacuous:
            print("\n  The symmetric filter removed nothing, so these two rows are the same")
            print("  comparison. C1 does not threaten the result -- not because filtering")
            print("  preserved it, but because the length constraint was already satisfied")
            print("  by every revision event.")
            if was_null:
                print(f"\n  The null stands (p = {as_published['p']:.4f}): divergence does not")
                print("  distinguish revisions from arbitrary text in the same traces.")
            else:
                print(f"\n  NOTE: this comparison is significant (p = {as_published['p']:.4f}).")
        elif was_null and now_null:
            print("\n  THE NULL SURVIVES symmetric length filtering, which here is a genuine")
            print("  robustness check. The published conclusion stands.")
        elif was_null and not now_null:
            print("\n  THE NULL DOES NOT SURVIVE. With both arms drawn from comparable trace")
            print("  lengths, revisions DO differ from random text. The published conclusion")
            print("  was an artifact of the asymmetric filter and must be withdrawn.")
        else:
            print("\n  Already significant as published; read the two rows directly.")
    if abs(results["length_vs_divergence"]["rho"]) > 0.1:
        print(f"\n  NOTE: divergence correlates with trace length "
              f"(rho = {results['length_vs_divergence']['rho']:+.3f}). That is the")
        print("  mechanism by which the asymmetric filter could bias the comparison, so")
        print("  the symmetric row above is the one to trust.")
    print("\n  C2 contamination, if any, is listed at the top -- fix before publishing.")

    json.dump(results, open(OUT, "w"), indent=2, default=str)
    print(f"\nWrote {OUT}")


if __name__ == "__main__":
    main()

---

# Part 3 — Arm-asymmetry checks (C3, C4, C5)

Leak-filtering cannot touch divergence — it never sees generated rollouts — but three structural asymmetries between the arms were identified and never tested. C3 is positional distribution: revision and filler differ in relative trace position (0.477 vs 0.515, p = 0.0006), and the random arm had never been compared at all; only medians were reported, never a test.

C4 and C5 are tested individually **and together** — removing each alone can leave the other explaining the result.


In [ ]:
# =============================================================================
# DIVERGENCE ARM-ASYMMETRY CHECKS (C3, C4, C5)
#
# Leak-filtering cannot affect divergence -- it never touches generated
# rollouts. But three structural asymmetries between the arms were identified
# and never tested. The revision-vs-random null is the load-bearing result, so
# they need closing rather than assuming.
#
#   C3  POSITIONAL DISTRIBUTION. Revision and filler differ in relative trace
#       position (0.477 vs 0.515, p = 0.0006 from the gap diagnostic). The
#       random arm was never compared. Only medians were reported, never a test.
#
#   C4  POSITIONAL BOUNDS. Random positions are drawn from
#       randrange(1000, len(fr) - 800), so they can never fall in the first
#       1000 or last 800 characters. Revision and filler events have no such
#       restriction. If divergence varies along a trace, the arms are sampling
#       different regions.
#
#   C5  MARKER-GAP ASYMMETRY. Revision and filler windows exclude the marker
#       text (extract_window is called with cs..ce spanning the marker), while
#       random uses a zero-width split (cs..cs) and excludes nothing. Revision
#       and filler before/after windows therefore have a ~5-15 char gap that
#       random windows do not.
#
# EACH IS TESTED DIRECTLY, and C4/C5 are tested by re-running the decisive
# comparison with the asymmetry REMOVED. If the null survives all three, the
# divergence conclusion is properly supported.
#
# COST: CPU + internet for the embedding model. Minutes.
# =============================================================================



from pathlib import Path
from collections import defaultdict
import json
import random
import re

import numpy as np
from scipy.stats import mannwhitneyu, ks_2samp, spearmanr

In [ ]:
from sentence_transformers import SentenceTransformer

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR = GDRIVE_BASE / "labels"
TRACES_DIR = GDRIVE_BASE / "traces"
REJECT_DIR = GDRIVE_BASE / "rejects"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
REPAIRED_GT = EXPORT_DIR / "repaired_ground_truth.json"

In [ ]:
OUT = EXPORT_DIR / "divergence_asymmetry_checks.json"

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED = 24601

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE, RAW_BUFFER_AFTER = 2500, 1200
N_BEFORE = N_AFTER = 6
RAND_LO, RAND_HI = 1000, 800          # the bounds under investigation (C4)

In [ ]:
MIN_LEN = 2000

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B", "DeepSeek-R1-Distill-Qwen-14B"]


def clean_bpe(t):
    return t.replace("\u0120", " ").replace("\u010a", "\n")


def split_sentences(text):
    return [p for p in re.split(r'(?<=[.!?])\s+', text.strip()) if p.strip()]


def take_by_budget(sents, budget, from_end):
    if not sents:
        return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s)
        total += len(s) + 1
        if total >= budget:
            break
    return list(reversed(out)) if from_end else out


def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE)
    re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_])
    ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)):
        return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]):
        after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after:
        return None, None
    return before, after


def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

In [ ]:
def eligible(tr):
    return tr is not None and tr.get("correct", False) and not tr.get("truncated", False)


def load_false_positives():
    if not REPAIRED_GT.exists():
        return set()
    rep = json.load(open(REPAIRED_GT))
    return {tuple(k.split("|", 1)) for k, v in rep.items()
            if v.get("correct_old") and not v.get("correct_new")}


def build(cache, rng, fps):
    """Revision and filler arms, plus THREE random variants:
       rnd_std   - as published: bounds (1000, len-800), zero-width split
       rnd_free  - C4 removed: bounds widened to the full usable range
       rnd_gap   - C5 removed: a marker-sized gap excised at the split point
    """
    rev, flr = [], []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if mdir.exists():
            for f in mdir.glob("*.json"):
                obj = json.load(open(f))
                pid = obj.get("problem_id", f.stem)
                if (model, pid) in fps:
                    continue
                revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
                if not revs:
                    continue
                tr = load_trace(model, pid, cache)
                if not eligible(tr):
                    continue
                fr = tr.get("full_response", "")
                for r in revs:
                    cs, ce = r.get("char_start"), r.get("char_end")
                    if cs is None or ce is None or not (0 <= cs < len(fr)):
                        continue
                    b, a = extract_window(fr, cs, ce)
                    if b is None:
                        continue
                    rev.append({"arm": "revision", "model": model, "problem_id": pid,
                                "before": b, "after": a, "trace_len": len(fr),
                                "rel_pos": cs / max(1, len(fr)),
                                "gap": ce - cs})
        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if rp.exists():
            for e in json.load(open(rp)):
                pid = e["problem_id"]
                if (model, pid) in fps:
                    continue
                tr = load_trace(model, pid, cache)
                if not eligible(tr):
                    continue
                fr = tr.get("full_response", "")
                cs, matched = e.get("char_start"), e.get("matched_text", "")
                if cs is None or not (0 <= cs < len(fr)):
                    continue
                ce = min(cs + len(matched), len(fr))
                b, a = extract_window(fr, cs, ce)
                if b is None:
                    continue
                flr.append({"arm": "filler", "model": model, "problem_id": pid,
                            "before": b, "after": a, "trace_len": len(fr),
                            "rel_pos": cs / max(1, len(fr)), "gap": ce - cs})

    per_model = defaultdict(int)
    for e in rev:
        per_model[e["model"]] += 1
    median_gap = int(np.median([e["gap"] for e in rev])) if rev else 8

    def make_random(tag, lo, hi, gap):
        out = []
        for model, n_want in per_model.items():
            pids = [pid for (m, pid), tr in cache.items()
                    if m == model and eligible(tr) and (m, pid) not in fps]
            if not pids:
                continue
            made = attempts = 0
            while made < n_want and attempts < n_want * 30:
                attempts += 1
                pid = pids[rng.randrange(len(pids))]
                fr = cache[(model, pid)].get("full_response", "")
                span_lo, span_hi = lo, len(fr) - hi - gap
                if span_hi <= span_lo:
                    continue
                cs = rng.randrange(span_lo, span_hi)
                b, a = extract_window(fr, cs, cs + gap)
                if b is None:
                    continue
                out.append({"arm": tag, "model": model, "problem_id": pid,
                            "before": b, "after": a, "trace_len": len(fr),
                            "rel_pos": cs / max(1, len(fr)), "gap": gap})
                made += 1
        return out

    rnd_std = make_random("random_std", RAND_LO, RAND_HI, 0)
    # C4 removed: allow the full range a revision could occupy
    rnd_free = make_random("random_free", 1, 1, 0)
    # C5 removed: excise a marker-sized gap, matching revision/filler windows
    rnd_gap = make_random("random_gap", RAND_LO, RAND_HI, median_gap)
    # BOTH removed. Tested separately above, but the asymmetries can interact --
    # each alone is small and they act in different directions, so the combined
    # case is not determined by the individual results.
    rnd_both = make_random("random_both", 1, 1, median_gap)
    return rev, flr, rnd_std, rnd_free, rnd_gap, rnd_both, median_gap


def mw(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 5 or len(b) < 5:
        return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    rng = np.random.RandomState(0)
    d = [np.median(rng.choice(a, len(a), True)) - np.median(rng.choice(b, len(b), True))
         for _ in range(3000)]
    return {"p": float(p), "median_a": float(np.median(a)), "median_b": float(np.median(b)),
            "median_diff": float(np.median(a) - np.median(b)),
            "ci95": [float(x) for x in np.percentile(d, [2.5, 97.5])],
            "n_a": len(a), "n_b": len(b)}


def show(tag, r):
    if r is None:
        print(f"    {tag:<44} (insufficient n)")
        return
    star = "*" if r["p"] < 0.05 else " "
    print(f"    {tag:<44} {r['median_a']:.4f} vs {r['median_b']:.4f}  "
          f"diff {r['median_diff']:+.4f} CI[{r['ci95'][0]:+.4f},{r['ci95'][1]:+.4f}] "
          f"p={r['p']:.4f}{star} n={r['n_a']}/{r['n_b']}")


def main():
    rng = random.Random(SEED)
    cache = {}
    fps = load_false_positives()
    print(f"\nExcluding {len(fps)} false-positive problems from all arms.")
    print("Building arms (3 random variants)...")
    rev, flr, rnd_std, rnd_free, rnd_gap, rnd_both, median_gap = build(cache, rng, fps)
    print(f"  revision {len(rev)}  filler {len(flr)}")
    print(f"  random: std {len(rnd_std)}  free {len(rnd_free)}  gap {len(rnd_gap)}  "
          f"both {len(rnd_both)}")
    print(f"  median revision marker gap: {median_gap} chars")
    results = {"n": {"revision": len(rev), "filler": len(flr),
                     "random_std": len(rnd_std), "random_free": len(rnd_free),
                     "random_gap": len(rnd_gap), "random_both": len(rnd_both)},
               "median_marker_gap": median_gap}

    # ---------------- C3/C4: positional distributions -----------------------
    print("\n" + "#" * 84)
    print("# C3/C4 — POSITIONAL DISTRIBUTIONS")
    print("#" * 84)
    pos = {name: np.array([e["rel_pos"] for e in arm], float)
           for name, arm in (("revision", rev), ("filler", flr),
                             ("random_std", rnd_std), ("random_free", rnd_free))}
    print(f"\n  {'arm':<14}{'n':>6}{'median':>9}{'p05':>8}{'p95':>8}{'frac<0.1':>10}")
    for name, P in pos.items():
        print(f"  {name:<14}{len(P):>6}{np.median(P):>9.3f}"
              f"{np.percentile(P, 5):>8.3f}{np.percentile(P, 95):>8.3f}"
              f"{(P < 0.1).mean():>10.1%}")
    print("\n  distribution tests vs revision (Kolmogorov-Smirnov):")
    c34 = {}
    for name in ("filler", "random_std", "random_free"):
        ks, p = ks_2samp(pos["revision"], pos[name])
        flag = "  <-- DIFFERS" if p < 0.01 else ""
        print(f"    revision vs {name:<14} KS={ks:.3f}  p={p:.3g}{flag}")
        c34[name] = {"ks": float(ks), "p": float(p)}
    results["positional_ks"] = c34

    # does divergence depend on position at all?
    print("\n  computing divergence...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)

    def diverge(ev):
        b = emb.encode(ev["before"], normalize_embeddings=True)
        a = emb.encode(ev["after"], normalize_embeddings=True)
        cb, ca = b.mean(axis=0), a.mean(axis=0)
        return 1.0 - float(np.dot(cb, ca) / (np.linalg.norm(cb) * np.linalg.norm(ca)))

    for arm in (rev, flr, rnd_std, rnd_free, rnd_gap, rnd_both):
        for e in arm:
            e["divergence"] = diverge(e)

    allev = rev + flr + rnd_std
    rho, prho = spearmanr([e["rel_pos"] for e in allev], [e["divergence"] for e in allev])
    print(f"\n  position vs divergence: Spearman rho = {rho:+.3f}, p = {prho:.3g}")
    results["position_vs_divergence"] = {"rho": float(rho), "p": float(prho)}
    if abs(rho) < 0.05:
        print("    -> divergence barely depends on position, so C3/C4 cannot bias the")
        print("       comparison much even where the distributions differ.")
    else:
        print("    -> divergence DOES depend on position; the random_free row below is")
        print("       the one to trust.")

    # ---------------- decisive comparisons ----------------------------------
    print("\n" + "#" * 84)
    print("# DECISIVE — revision vs random, with each asymmetry removed in turn")
    print("#" * 84 + "\n")
    d_rev = [e["divergence"] for e in rev]
    r_std = mw(d_rev, [e["divergence"] for e in rnd_std])
    r_free = mw(d_rev, [e["divergence"] for e in rnd_free])
    r_gap = mw(d_rev, [e["divergence"] for e in rnd_gap])
    r_both = mw(d_rev, [e["divergence"] for e in rnd_both])
    show("as published (bounds 1000/800, no gap)", r_std)
    show("C4 removed (full positional range)", r_free)
    show(f"C5 removed (marker gap of {median_gap} chars)", r_gap)
    show("BOTH removed (C4 + C5 together)", r_both)
    results["decisive"] = {"as_published": r_std, "c4_removed": r_free,
                           "c5_removed": r_gap, "both_removed": r_both}

    # ---------------- read ---------------------------------------------------
    print("\n" + "#" * 84)
    print("# READ")
    print("#" * 84)
    verdicts = {"as published": r_std, "C4 removed": r_free,
                "C5 removed": r_gap, "BOTH removed": r_both}
    sig = {k: (v is not None and v["p"] < 0.05) for k, v in verdicts.items()}
    print()
    for k, v in verdicts.items():
        if v:
            print(f"  {k:<16} p = {v['p']:.4f}  ->  {'SIGNIFICANT' if sig[k] else 'null'}")
    if not any(sig.values()):
        print("\n  THE NULL HOLDS under every variant, INCLUDING both asymmetries removed")
        print("  together. Neither the positional bounds nor the marker-gap asymmetry")
        print("  drives the result, alone or in combination. Together with the length")
        print("  filter being vacuous and the false positives excluded, the divergence")
        print("  conclusion is now supported rather than assumed:")
        print("  divergence does not distinguish revisions from arbitrary text.")
    else:
        broken = [k for k, s in sig.items() if s]
        print(f"\n  THE NULL BREAKS under: {', '.join(broken)}")
        print("  At least one asymmetry was masking a real difference. The published")
        print("  conclusion must be withdrawn and restated using the corrected variant.")

    json.dump(results, open(OUT, "w"), indent=2, default=str)
    print(f"\nWrote {OUT}")


if __name__ == "__main__":
    main()

---

# Part 4 — Why is the filler arm low?

Part 1 found revision ≈ random ≫ filler. So the question is not why revisions are high — they are ordinary — but why **fillers are low**.

This part tests local repetition as the explanation, stratifying by a `max_pair_sim` covariate. The headline finding is that the revision-filler gap largely vanishes once local repetition is controlled, and that the `alternatively` marker's apparent effect is a repetition effect. It also documents a matching flaw in part 1's filler arm.


In [ ]:
# =============================================================================
# WHY IS THERE A REVISION-FILLER DIVERGENCE GAP?
#
# Observed (divergence_control.py):
#   revision 0.2981  ~=  random 0.2931  >>  filler 0.2631
# Revisions are indistinguishable from arbitrary text, but fillers sit well
# BELOW both. So the question is not "why are revisions high" -- it is
# "why are fillers LOW".
#
# ** FIRST, A FLAW IN THE ORIGINAL CONTROL. ** In divergence_control.py the
# filler arm was matched down to revision demand (1262 events), but the
# revision arm was NOT correspondingly restricted -- it kept all 2649 events,
# including 1387 in (model, marker) cells with no filler counterpart at all.
# The two arms therefore had different marker compositions and the comparison
# was not truly matched. H1 below tests whether the gap survives proper
# matching. If it does not, the gap is a composition artifact and there is
# nothing further to explain.
#
# HYPOTHESES TESTED
#   H1 Composition   - gap vanishes under symmetric matching / within-marker
#                      stratification -> artifact of unmatched marker mix.
#   H2 Position      - fillers sit at systematically different points in the
#                      trace (e.g. early setup, where reasoning is homogeneous).
#   H3 Window size   - fillers get fewer sentences (noisier centroids, or
#                      truncated windows near trace boundaries).
#   H4 Text length   - before/after text lengths differ systematically.
#   H5 Repetition    - fillers sit in locally repetitive stretches, so
#                      before/after are near-duplicates. Measured directly as
#                      max pairwise similarity between before and after
#                      sentences.
#
# Anything surviving all of these is evidence for the substantive reading:
# fillers genuinely mark points of unusual continuity in the reasoning.
#
# COST: CPU + internet for embeddings. Minutes.
# =============================================================================



from pathlib import Path
from collections import defaultdict, Counter
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

In [ ]:
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
REJECT_DIR  = GDRIVE_BASE / "rejects"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH    = EXPORT_DIR / "divergence_gap_diagnostic.json"

In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED = 4242

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE,  RAW_BUFFER_AFTER  = 2500, 1200
N_BEFORE = N_AFTER = 6

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

In [ ]:
# ---------------------------------------------------------------- helpers ---
def clean_bpe(t):
    return t.replace("\u0120", " ").replace("\u010a", "\n")


def split_sentences(text):
    return [p for p in re.split(r'(?<=[.!?])\s+', text.strip()) if p.strip()]


def take_by_budget(sents, budget, from_end):
    if not sents:
        return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget:
            break
    return list(reversed(out)) if from_end else out


def normalize_marker(t):
    return re.sub(r"[^\w\s]", "", str(t).lower()).strip()


def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE)
    re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_])
    ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)):
        return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after  = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]):
        after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after:
        return None, None
    return before, after


def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

In [ ]:
def build_arms(cache):
    rev, flr = [], []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if mdir.exists():
            for f in mdir.glob("*.json"):
                obj = json.load(open(f))
                pid = obj.get("problem_id", f.stem)
                revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
                if not revs:
                    continue
                tr = load_trace(model, pid, cache)
                if tr is None or not tr.get("correct", False) or tr.get("truncated", False):
                    continue
                fr = tr.get("full_response", "")
                for r in revs:
                    cs, ce = r.get("char_start"), r.get("char_end")
                    if cs is None or ce is None or not (0 <= cs < len(fr)):
                        continue
                    b, a = extract_window(fr, cs, ce)
                    if b is None:
                        continue
                    rev.append({"arm": "revision", "model": model, "problem_id": pid,
                                "marker": normalize_marker(r.get("matched_text", "")),
                                "before": b, "after": a,
                                "rel_pos": cs / max(1, len(fr))})
        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if rp.exists():
            for e in json.load(open(rp)):
                pid = e["problem_id"]
                tr = load_trace(model, pid, cache)
                if tr is None or not tr.get("correct", False) or tr.get("truncated", False):
                    continue
                fr = tr.get("full_response", "")
                cs, matched = e.get("char_start"), e.get("matched_text", "")
                if cs is None or not (0 <= cs < len(fr)):
                    continue
                b, a = extract_window(fr, cs, min(cs + len(matched), len(fr)))
                if b is None:
                    continue
                flr.append({"arm": "filler", "model": model, "problem_id": pid,
                            "marker": normalize_marker(matched),
                            "before": b, "after": a,
                            "rel_pos": cs / max(1, len(fr))})
    return rev, flr


def mw(a, b):
    if len(a) < 3 or len(b) < 3:
        return None, None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return float(p), float(u / (len(a) * len(b)))


def main():
    cache = {}
    print("Building arms...")
    rev, flr = build_arms(cache)
    print(f"  revision {len(rev)}   filler {len(flr)}")

    print(f"\nLoading {EMBED_MODEL_NAME} ...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)

    def annotate(ev):
        B = emb.encode(ev["before"], normalize_embeddings=True)
        A = emb.encode(ev["after"],  normalize_embeddings=True)
        cb, ca = B.mean(axis=0), A.mean(axis=0)
        ev["divergence"] = 1.0 - float(np.dot(cb, ca) /
                                        (np.linalg.norm(cb) * np.linalg.norm(ca)))
        ev["n_before"], ev["n_after"] = len(ev["before"]), len(ev["after"])
        ev["len_before"] = sum(len(s) for s in ev["before"])
        ev["len_after"]  = sum(len(s) for s in ev["after"])
        # H5: strongest before/after sentence-pair similarity -> near-duplication
        ev["max_pair_sim"] = float((B @ A.T).max())

    print("Computing divergence + covariates...")
    for e in rev + flr:
        annotate(e)

    results = {}
    v_rev_all = np.array([e["divergence"] for e in rev])
    v_flr_all = np.array([e["divergence"] for e in flr])
    p_all, auc_all = mw(v_rev_all, v_flr_all)
    print(f"\n{'='*72}\nBASELINE (unmatched, as in the original control)\n{'='*72}")
    print(f"  revision n={len(v_rev_all)} median {np.median(v_rev_all):.4f}")
    print(f"  filler   n={len(v_flr_all)} median {np.median(v_flr_all):.4f}")
    print(f"  diff {np.median(v_rev_all)-np.median(v_flr_all):+.4f}   p={p_all:.3g}")
    results["baseline"] = {"n_rev": len(v_rev_all), "n_flr": len(v_flr_all),
                           "median_rev": float(np.median(v_rev_all)),
                           "median_flr": float(np.median(v_flr_all)),
                           "p": p_all}

    # ---------------------------------------------------------------- H1 ---
    # Symmetric matching: for each (model, marker) cell take k = min(n_rev,
    # n_flr) from BOTH arms. Marker composition then identical by construction.
    print(f"\n{'='*72}\nH1 — COMPOSITION: symmetric matching on (model, marker)\n{'='*72}")
    rng = random.Random(SEED)
    cells_rev, cells_flr = defaultdict(list), defaultdict(list)
    for e in rev: cells_rev[(e["model"], e["marker"])].append(e)
    for e in flr: cells_flr[(e["model"], e["marker"])].append(e)

    m_rev, m_flr = [], []
    for cell in set(cells_rev) & set(cells_flr):
        k = min(len(cells_rev[cell]), len(cells_flr[cell]))
        m_rev.extend(rng.sample(cells_rev[cell], k))
        m_flr.extend(rng.sample(cells_flr[cell], k))
    a, b = np.array([e["divergence"] for e in m_rev]), np.array([e["divergence"] for e in m_flr])
    p_m, auc_m = mw(a, b)
    print(f"  symmetric-matched n={len(a)} per arm (identical marker composition)")
    print(f"  revision median {np.median(a):.4f}   filler median {np.median(b):.4f}")
    print(f"  diff {np.median(a)-np.median(b):+.4f}   p={p_m:.3g}   AUC={auc_m:.3f}")
    results["h1_symmetric_matched"] = {"n_per_arm": len(a),
        "median_rev": float(np.median(a)), "median_flr": float(np.median(b)),
        "p": p_m, "auc": auc_m}

    print("\n  within-marker breakdown (cells with >=20 events per arm):")
    per_marker = {}
    for cell in sorted(set(cells_rev) & set(cells_flr)):
        r_ = [e["divergence"] for e in cells_rev[cell]]
        f_ = [e["divergence"] for e in cells_flr[cell]]
        if len(r_) >= 20 and len(f_) >= 20:
            p_c, _ = mw(r_, f_)
            print(f"    {cell[1]!r:<22} {cell[0][-12:]:<12} "
                  f"rev {np.median(r_):.4f} (n={len(r_):>4})  "
                  f"fil {np.median(f_):.4f} (n={len(f_):>4})  "
                  f"diff {np.median(r_)-np.median(f_):+.4f}  p={p_c:.3g}")
            per_marker[f"{cell[0]}|{cell[1]}"] = {
                "median_rev": float(np.median(r_)), "median_flr": float(np.median(f_)),
                "n_rev": len(r_), "n_flr": len(f_), "p": p_c}
    results["h1_per_marker"] = per_marker

    # ------------------------------------------------------- H2..H5 -------
    print(f"\n{'='*72}\nH2-H5 — COVARIATES (symmetric-matched sets)\n{'='*72}")
    cov = {}
    for name, key in [("H2 relative position in trace", "rel_pos"),
                      ("H3 n_before sentences", "n_before"),
                      ("H3 n_after sentences", "n_after"),
                      ("H4 before text length", "len_before"),
                      ("H4 after text length", "len_after"),
                      ("H5 max before/after sentence similarity", "max_pair_sim")]:
        ra = np.array([e[key] for e in m_rev], dtype=float)
        fa = np.array([e[key] for e in m_flr], dtype=float)
        p_c, auc_c = mw(ra, fa)
        flag = "  <-- DIFFERS" if (p_c is not None and p_c < 0.01) else ""
        print(f"  {name:<42} rev {np.median(ra):>8.4f}   fil {np.median(fa):>8.4f}   "
              f"p={p_c:.3g}{flag}")
        cov[key] = {"median_rev": float(np.median(ra)), "median_flr": float(np.median(fa)),
                    "p": p_c, "auc": auc_c}
    results["covariates"] = cov

    # ------------------------------------------ H5 follow-up: partial ------
    # If max_pair_sim differs, check whether the divergence gap survives
    # restricting to events with COMPARABLE max_pair_sim (coarse stratification).
    print(f"\n{'='*72}\nH5 FOLLOW-UP — divergence gap within max_pair_sim strata\n{'='*72}")
    allsim = np.array([e["max_pair_sim"] for e in m_rev + m_flr])
    edges = np.percentile(allsim, [0, 25, 50, 75, 100])
    strata = {}
    for i in range(4):
        lo, hi = edges[i], edges[i+1]
        ra = [e["divergence"] for e in m_rev if lo <= e["max_pair_sim"] <= hi]
        fa = [e["divergence"] for e in m_flr if lo <= e["max_pair_sim"] <= hi]
        if len(ra) >= 20 and len(fa) >= 20:
            p_s, _ = mw(ra, fa)
            print(f"  sim [{lo:.3f},{hi:.3f}]  rev {np.median(ra):.4f} (n={len(ra):>4})  "
                  f"fil {np.median(fa):.4f} (n={len(fa):>4})  "
                  f"diff {np.median(ra)-np.median(fa):+.4f}  p={p_s:.3g}")
            strata[f"{lo:.3f}-{hi:.3f}"] = {
                "median_rev": float(np.median(ra)), "median_flr": float(np.median(fa)),
                "n_rev": len(ra), "n_flr": len(fa), "p": p_s}
    results["h5_strata"] = strata

    # ------------------------------------------------------------- read ---
    print(f"\n{'#'*72}\n# READ\n{'#'*72}")
    if p_m is None:
        print("Symmetric matching left too few events. Inconclusive.")
    elif p_m >= 0.05:
        print("H1 SUPPORTED -- the gap DISAPPEARS under symmetric matching.")
        print("The original revision-filler difference was an artifact of unmatched")
        print("marker composition (the filler arm was matched down to revision demand,")
        print("but the revision arm kept 1387 events in cells with no filler counterpart).")
        print("There is then nothing substantive to explain: divergence does not")
        print("distinguish revisions from fillers OR from random text. Update the")
        print("divergence section accordingly -- this strengthens the null.")
    else:
        print(f"H1 NOT sufficient -- gap survives symmetric matching "
              f"({np.median(a)-np.median(b):+.4f}, p={p_m:.3g}).")
        print("Composition alone does not explain it. Check the covariate table above:")
        print("  - any covariate flagged 'DIFFERS' is a live alternative explanation;")
        print("  - if max_pair_sim differs AND the gap shrinks/vanishes within its strata,")
        print("    the filler-continuity reading is supported (fillers sit where")
        print("    before/after text is near-duplicated).")
        print("  - if the gap survives every stratum with no covariate differing, the")
        print("    effect is substantive and currently unexplained -- report it as such.")

    json.dump(results, open(OUT_PATH, "w"), indent=2)
    print(f"\nWrote {OUT_PATH}")


if __name__ == "__main__":
    main()

---

## Reading the output

The question that matters is **does revision exceed RANDOM** — not whether it exceeds filler. Beating filler alone does not establish revision-sensitivity, because fillers are marker words at points where by construction nothing changes, which makes them an artificially low baseline rather than a neutral one.

Check CI widths before calling any null conclusive, and quote part 2's figures rather than part 1's.
